In [15]:
#imports
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import importlib
import sys
import os
from astropy.cosmology import Planck18 as cosmo




In [16]:
# --- User toggle ---
Cristina = True
Shar = not Cristina 

# --- System Configurations ---

if Cristina:
    print("[CONFIG] Using Cristina's local MacBook setup")
    sys.path.insert(0, "/Users/andradenebula/Documents/Research/Transient_Metrics/Multi_Transient_Metrics_Hub")
    os.environ["RUBIN_SIM_DATA_DIR"] = "/Users/andradenebula/rubin_sim_data"
    db_dir = "/Users/andradenebula/Documents/Research/Transient_Metrics/Multi_Transient_Metrics_Hub"

elif Shar:
    print("[CONFIG] Using Shar's Dirac server setup")
    sys.path.insert(0, "/lustre/lrspec/metrics")
    sys.path.insert(0, "/home/3155/metrics/Multi_Transient_Metrics_Hub")
    os.environ["RUBIN_SIM_DATA_DIR"] = "/lustre/lrspec/metrics/rubin_sim_data"
    db_dir = "/lustre/lrspec/metrics"

# Shared config
sys.path.append(os.path.abspath(".."))  # For shared_utils


[CONFIG] Using Cristina's local MacBook setup


In [17]:
#this happens twice because idk why but it only works like that
#...you still have to run it twice if it gives warnings
# C: Make sure you save your metric file before running. 

metric_filename = "local_LFBOTmetric"
s_u = "shared_utils"

# --- Reload metric module ---
if metric_filename in sys.modules:
    del sys.modules[metric_filename]
metric = __import__(metric_filename)
importlib.reload(metric)

# --- Reload shared_utils module ---
if s_u in sys.modules:
    del sys.modules[s_u]
shared_utils = __import__(s_u)
importlib.reload(shared_utils)

print(f"[INFO] Loaded metric module: {metric_filename}")
print(f"[INFO] Loaded shared_utils module")




[INFO] Loaded metric module: local_LFBOTmetric
[INFO] Loaded shared_utils module


In [18]:
#metric configurations

#control whether we generate new files
generate_new_templates = True
generate_new_pop = True
make_debug_plots = False #toggle whether or not the pop generation makes plots

#population variables
rate_density = 420e-9

# apply None for non-use case 
z_min, z_max = None, None
dmin, dmax = 10, 600

#other
gal_lat_cut = None #latitude cut, for Galactic phenomena
t_start = 1 #start time in days
t_end = 3652

# Whether to remove Metric_temp_* folders after running
clean_temp = True  # <- NEW toggle

#cadence variables
cadences = ['four_roll_v4.3.1_10yrs', 'baseline_v4.3.1_10yrs']
ignore_triples = False #turn this to true to ignore triples
filters = ['g', 'r'] #doesn't work rn i think but
#if we wanted to look at less filters then we would adjust that here

# Standardized output paths for this science case
paths = metric.get_output_paths(case_label="LFBOTs")  # <- can change to 'KNe' etc.

storage_dir = paths['storage_dir']
templates_file = paths['templates_file']
pop_file = paths['pop_file']


In [19]:
#load and/or generate light curves
shared_lc_model = metric.load_or_generate_templates(
    templates_file=templates_file,
    generate_new=generate_new_templates
)

[INFO] Generating 1000 light curve templates.


TypeError: load_or_generate_templates() got an unexpected keyword argument 'save_to'

In [20]:
#plot light curves from pkl file if desired
shared_utils.plot_some_lcs_from_pkl(templates_file, num=3)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/andradenebula/Documents/Research/Transient_Metrics/Multi_Transient_Metrics_Hub/output/LFBOTs/LFBOTs_templates.pkl'

In [21]:
# Load or generate population slicer
slicer = metric.load_or_generate_population(
    t_start=t_start,
    t_end=t_end,
    d_min=dmin,
    d_max=dmax,
    z_min=z_min,
    z_max=z_max,
    seed=42,
    num_lightcurves=1000,
    gal_lat_cut=gal_lat_cut,
    rate_density=rate_density,
    pop_file=pop_file,
    generate_new=generate_new_pop,
    make_debug_plots=make_debug_plots
)


TypeError: load_or_generate_population() got an unexpected keyword argument 'z_min'

## All 10 years

In [22]:
#run detection metric
df_obs_arr = shared_utils.run_detect(metric, slicer, cadences, shared_lc_model, db_dir, storage_dir, debug=True, plot=True, clean_temp=clean_temp)

NameError: name 'slicer' is not defined

In [24]:
#choose what to run in run_multi_metrics
# we can remove historical if we want
multi_metrics = metric.get_multi_metrics(shared_lc_model, include=['detect', 'characterize'])


NameError: name 'shared_lc_model' is not defined

In [25]:
shared_utils.run_multi_metrics(multi_metrics, slicer, cadences, shared_lc_model, db_dir, storage_dir, ignore_triples=False, plot=True, clean_temp=clean_temp)

NameError: name 'multi_metrics' is not defined

## Tests

In [ ]:
print("Type:", type(lightcurves))
if isinstance(lightcurves, dict):
    print("Top-level keys:", list(lightcurves.keys()))
    for key in lightcurves:
        print(f"\nKey: {key} → Type: {type(lightcurves[key])}")
        try:
            # Show subkeys if dict
            print("  Subkeys:", list(lightcurves[key].keys())[:3])
        except Exception as e:
            print("  Could not inspect subkeys:", e)
else:
    print("First item:", lightcurves[0])


In [ ]:
# Look inside one of the filter light curves
event_idx = 0
event_lc = lightcurves[event_idx]

# Pick a filter you know exists, like 'r'
if 'r' in event_lc:
    print("event_lc['r'] type:", type(event_lc['r']))
    print("event_lc['r'] keys:", event_lc['r'].keys())
    print("event_lc['r'] sample:", list(event_lc['r'].items())[:3])
else:
    print("'r' band not found in this event.")


In [ ]:
# Load the light curves
with open(templates_file, 'rb') as f:
    templates = pickle.load(f)

# Extract the light curve list
lightcurves = templates['lightcurves']

# Choose event index
event_idx = 0
event_lc = lightcurves[event_idx]  # this is a dict with keys: u,g,r,...

# Set filter names and colors
filters = ['u', 'g', 'r', 'i', 'z', 'y']
colors = ['purple', 'green', 'red', 'orange', 'brown', 'blue']

# Plot
plt.figure(figsize=(10, 6))

for f_name, color in zip(filters, colors):
    if f_name in event_lc:
        ph = event_lc[f_name]['ph']
        mag = event_lc[f_name]['mag']
        plt.plot(ph, mag, label=f"{f_name}-band", color=color)

plt.gca().invert_yaxis()
plt.xlabel("Time [days]")
plt.ylabel("Apparent Magnitude")
plt.title(f"Light Curve for Event #{event_idx} in All Filters")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()
